<a href="https://colab.research.google.com/github/NABI-SNU/book/blob/main/tutorials/Session_1_DeepLearning/student/Tutorial1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Tutorial 1: From Linear Models to Neural Networks (MNIST)

**Session 1: Deep Learning Foundations**

**Objective:** Build intuition for why neural networks extend linear models.


## Tutorial Objectives

By the end of this tutorial, you will be able to:

- Explain a linear classifier of the form

$$
\mathbf{y} = \mathbf{W}\mathbf{x} + \mathbf{b}
$$

- Explain why inserting a non-linear activation changes what the model can represent:

$$
\mathbf{h} = f(\mathbf{W}\mathbf{x} + \mathbf{b})
$$

- Train logistic regression and a one-hidden-layer neural network on MNIST.
- Compare linear and non-linear models using accuracy and misclassified examples.


In [ ]:
# Imports and shared settings
import random
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

SEED = 4
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

plt.rcParams['figure.figsize'] = (6, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False


In [ ]:
def get_mnist_loaders(batch_size=128, train_subset=12000, val_size=2000):
    """Return small MNIST train/validation/test loaders for quick tutorials."""
    transform = transforms.ToTensor()

    full_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

    if train_subset is not None:
        indices = torch.randperm(len(full_train))[:train_subset + val_size]
        full_train = Subset(full_train, indices)

    train_size = len(full_train) - val_size
    train_data, val_data = random_split(
        full_train,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED),
    )

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = get_mnist_loaders()
images, labels = next(iter(train_loader))
print('Image batch:', images.shape)
print('Label batch:', labels.shape)


## Inspect MNIST

MNIST images are grayscale handwritten digits. Each image has shape `1 x 28 x 28`: one channel, height 28, width 28.
A linear classifier flattens this image into a vector with 784 entries.


In [ ]:
fig, axs = plt.subplots(2, 6, figsize=(9, 3))
for ax, image, label in zip(axs.ravel(), images[:12], labels[:12]):
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(f'label: {label.item()}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Model 1: Logistic Regression

For MNIST classification, logistic regression is a linear map from pixels to 10 class scores, also called logits:

$$
\mathbf{z} = \mathbf{W}\mathbf{x} + \mathbf{b}, \quad \mathbf{z} \in \mathbb{R}^{10}.
$$

PyTorch's `CrossEntropyLoss` combines `log_softmax` and negative log likelihood, so the model should return raw logits rather than probabilities.


In [ ]:
class LogisticRegressionMNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 10),
        )

    def forward(self, x):
        return self.net(x)

linear_model = LogisticRegressionMNIST()
print(linear_model)


In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model(images)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += images.size(0)

    return total_loss / total_examples, total_correct / total_examples


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(images)
            loss = loss_fn(logits, labels)

            total_loss += loss.item() * images.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_examples += images.size(0)

    return total_loss / total_examples, total_correct / total_examples


def fit(model, train_loader, val_loader, n_epochs=5, lr=1e-2, momentum=0.0):
    model = model.to(DEVICE)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(n_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(
            f'Epoch {epoch + 1:02d} | '
            f'train loss {train_loss:.3f}, acc {train_acc:.3f} | '
            f'val loss {val_loss:.3f}, acc {val_acc:.3f}'
        )

    return history


def plot_history(history, title='Training curves'):
    epochs = np.arange(1, len(history['train_loss']) + 1)
    fig, axs = plt.subplots(1, 2, figsize=(11, 4))

    axs[0].plot(epochs, history['train_loss'], marker='o', label='train')
    axs[0].plot(epochs, history['val_loss'], marker='o', label='validation')
    axs[0].set_xlabel('Epoch')
    axs[0].set_ylabel('Cross-entropy loss')
    axs[0].set_title('Loss')
    axs[0].legend()

    axs[1].plot(epochs, history['train_acc'], marker='o', label='train')
    axs[1].plot(epochs, history['val_acc'], marker='o', label='validation')
    axs[1].set_xlabel('Epoch')
    axs[1].set_ylabel('Accuracy')
    axs[1].set_ylim(0, 1)
    axs[1].set_title('Accuracy')
    axs[1].legend()

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


In [ ]:
linear_history = fit(linear_model, train_loader, val_loader, n_epochs=5, lr=0.1)
plot_history(linear_history, title='Logistic regression on MNIST')


## Model 2: One-Hidden-Layer Neural Network

A one-hidden-layer network first maps pixels into hidden features, applies a non-linearity, and then maps those features to class logits:

$$
\mathbf{h} = \mathrm{ReLU}(\mathbf{W}_1\mathbf{x} + \mathbf{b}_1), \quad
\mathbf{z} = \mathbf{W}_2\mathbf{h} + \mathbf{b}_2.
$$

The ReLU activation makes the model non-linear. This lets it learn decision boundaries that cannot be represented by a single linear classifier.


In [ ]:
class OneHiddenLayerNN(nn.Module):
    def __init__(self, hidden_units=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, 10),
        )

    def forward(self, x):
        return self.net(x)

nn_model = OneHiddenLayerNN(hidden_units=128)
print(nn_model)


In [ ]:
nn_history = fit(nn_model, train_loader, val_loader, n_epochs=5, lr=0.1)
plot_history(nn_history, title='One-hidden-layer neural network on MNIST')


## Compare the Two Models

The neural network usually improves over the linear classifier because it can learn intermediate features, not just one linear decision boundary per class.


In [ ]:
loss_fn = nn.CrossEntropyLoss()
linear_test_loss, linear_test_acc = evaluate(linear_model, test_loader, loss_fn)
nn_test_loss, nn_test_acc = evaluate(nn_model, test_loader, loss_fn)

print(f'Linear test accuracy: {linear_test_acc:.3f}')
print(f'1-layer NN test accuracy: {nn_test_acc:.3f}')


## Exercise: Visualize Misclassified Digits

Run the next cell for either model. What kinds of digits are confused? Are the errors ambiguous to you as a human?


In [ ]:
def plot_misclassified(model, loader, n_examples=12):
    model.eval()
    examples = []
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(DEVICE)).cpu()
            preds = logits.argmax(dim=1)
            wrong = preds != labels
            for image, label, pred in zip(images[wrong], labels[wrong], preds[wrong]):
                examples.append((image, label.item(), pred.item()))
                if len(examples) >= n_examples:
                    break
            if len(examples) >= n_examples:
                break

    fig, axs = plt.subplots(2, 6, figsize=(9, 3))
    for ax, (image, label, pred) in zip(axs.ravel(), examples):
        ax.imshow(image.squeeze(), cmap='gray')
        ax.set_title(f'true {label}, pred {pred}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

plot_misclassified(nn_model, test_loader)
